In [201]:
import pandas as pd
import numpy as np

In [202]:
patients=pd.read_csv('Datasets/patients.csv')
treatments=pd.read_csv('Datasets/treatments.csv')
adverse_reactions=pd.read_csv('Datasets/adverse_reactions.csv')
treatments_cut=pd.read_csv('Datasets/treatments_cut.csv')

In [203]:
# view dataset
patients.head()
patients['zip_code']=patients['zip_code'].astype(str)

In [204]:
treatments.head()

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,NaN
1,elliot,richardson,-,40u - 45u,7.56,7.09,0.97
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,NaN
3,skye,gormanston,33u - 36u,-,7.97,7.62,0.35
4,alissa,montez,-,33u - 29u,7.78,7.46,0.32


In [205]:
treatments_cut.head()

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,jožka,resanovič,22u - 30u,-,7.56,7.22,0.34
1,inunnguaq,heilmann,57u - 67u,-,7.85,7.45,NaN
2,alwin,svensson,36u - 39u,-,7.78,7.34,NaN
3,thể,lương,-,61u - 64u,7.64,7.22,0.92
4,amanda,ribeiro,36u - 44u,-,7.85,7.47,0.38


In [206]:
adverse_reactions.head()

,given_name,surname,adverse_reaction
0,berta,napolitani,injection site discomfort
1,lena,baer,hypoglycemia
2,joseph,day,hypoglycemia
3,flavia,fiorentino,cough
4,manouck,wubbels,throat irritation


Manual assessment

In [207]:
with pd.ExcelWriter('Exported_Data/clinical_trails.xlsx') as writer:
    patients.to_excel(writer,sheet_name='patients')
    treatments.to_excel(writer,sheet_name='treatments')
    treatments_cut.to_excel(writer,sheet_name='treatments_cut')
    adverse_reactions.to_excel(writer,sheet_name='adverse_reactions')

Issue with data
- 

Dirty Data - 
1. Table - `patients`
- patient_id=9 has misspelled name 'dsvid' instead of 'david'. `accuracy issue`
- State col sometimes contains full name and somtimes abbrivietation. `consistency`
- zip code col has entry with 4 digit. `validity issue`
- Data Missing for 12 patients in address, city, state, zip_code, country, contact `completion issue`
- incorrect data type assigned to assigned_sex, zip_code, birthdate `validity issue`
- duplicate entries by the name of john doe `accuracy issue`
- one patient has weigth = 48 pounds `accuracy issue`
- one patient have height = 27 inches `accuracy issue`

2. Table - `treatments` & `Treatments_cut`
- given_name and surname col is in all lower case. `consistency issue`
- remove u from auralin and novadra column. `validity issue`
- `-` in novardra and auralin col treated as nan `validity issue`
- missing value in hba1c change column `completion issue`
- 1 duplicated entry by the name josep day `accuracy issue`
- in hba1c_change 9 instead of 4 `accuracy issue`

3. Table - `Adverse_reactions`
- given_name and surname are all in lower case. `consistency issue`

Messy Data -
1. Table - `patients`
- contact col contain both phone number and email

2. Table - `Treatments` & `Treatments_cut`
- Auralin and novadra col should be split into 2 cols start and end dose
- Merge both the tables


3. Table - `adverse_reactions`
- This table should not exist independently.

Automatic  Assessment
- 
- Head and tail
- sample
- info
- isnull
- duplicated
- describe

In [208]:
# Checking of data biasness
patients.head()
patients.tail()
patients.sample(5)

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
203,204,female,Mùi,Lương,1778 Rodney Street,Harvester,MO,63301.0,United States,636-442-6946LuongHongMui@einrot.com,2/29/1956,192.7,60,37.6
450,451,male,Clinton,Miller,901 Southern Street,Roslyn,NY,11576.0,United States,516-626-8021ClintonKMiller@rhyta.com,8/16/1985,195.4,74,25.1
35,36,female,Kamila,Pecinová,3558 Longview Avenue,New York,New York,10004.0,United States,718-501-0503KamilaPecinova@dayrep.com,12/23/1985,198.9,62,36.4
284,285,male,Nilton,Quintanilla,4038 Farland Street,Walpole,MA,2081.0,United States,774-219-3140NiltonQuintanillaAlmonte@rhyta.com,2/9/1980,186.3,75,23.3
275,276,male,Eddie,Archer,2043 Jadewood Drive,Lombard,Illinois,60148.0,United States,EddieAArcher@gustr.com+1 (224) 305-6805,7/17/1982,158.6,69,23.4


In [209]:
# Column containing null and data_type of column
patients.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    str    
 2   given_name    503 non-null    str    
 3   surname       503 non-null    str    
 4   address       491 non-null    str    
 5   city          491 non-null    str    
 6   state         491 non-null    str    
 7   zip_code      491 non-null    str    
 8   country       491 non-null    str    
 9   contact       491 non-null    str    
 10  birthdate     503 non-null    str    
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(2), int64(2), str(10)
memory usage: 111.3 KB


In [210]:
patients[patients['address'].isna()]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
209,210,female,Lalita,Eldarkhanov,NaN,NaN,NaN,NaN,NaN,NaN,8/14/1950,143.4,62,26.2
219,220,male,Mỹ,Quynh,NaN,NaN,NaN,NaN,NaN,NaN,4/9/1978,237.8,69,35.1
230,231,female,Elisabeth,Knudsen,NaN,NaN,NaN,NaN,NaN,NaN,9/23/1976,165.9,63,29.4
234,235,female,Martina,Tománková,NaN,NaN,NaN,NaN,NaN,NaN,4/7/1936,199.5,65,33.2
242,243,male,John,O'Brian,NaN,NaN,NaN,NaN,NaN,NaN,2/25/1957,205.3,74,26.4
249,250,male,Benjamin,Mehler,NaN,NaN,NaN,NaN,NaN,NaN,10/30/1951,146.5,69,21.6
257,258,male,Jin,Kung,NaN,NaN,NaN,NaN,NaN,NaN,5/17/1995,231.7,69,34.2
264,265,female,Wafiyyah,Asfour,NaN,NaN,NaN,NaN,NaN,NaN,11/3/1989,158.6,63,28.1
269,270,female,Flavia,Fiorentino,NaN,NaN,NaN,NaN,NaN,NaN,10/9/1937,175.2,61,33.1
278,279,female,Generosa,Cabán,NaN,NaN,NaN,NaN,NaN,NaN,12/16/1962,124.3,69,18.4


In [211]:
treatments.info()

<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    280 non-null    str    
 1   surname       280 non-null    str    
 2   auralin       280 non-null    str    
 3   novodra       280 non-null    str    
 4   hba1c_start   280 non-null    float64
 5   hba1c_end     280 non-null    float64
 6   hba1c_change  171 non-null    float64
dtypes: float64(3), str(4)
memory usage: 21.8 KB


In [212]:
adverse_reactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   given_name        34 non-null     str  
 1   surname           34 non-null     str  
 2   adverse_reaction  34 non-null     str  
dtypes: str(3)
memory usage: 1.8 KB


In [213]:
patients['patient_id'].duplicated().sum()

np.int64(0)

In [214]:

patients[patients.duplicated(subset=['given_name','surname'])]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
229,230,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
237,238,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
244,245,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
251,252,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4
277,278,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,johndoe@email.com1234567890,1/1/1975,180.0,72,24.4


In [215]:
treatments[treatments.duplicated()]

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
136,joseph,day,29u - 36u,-,7.7,7.19,NaN


In [216]:
treatments[treatments.duplicated(subset=['given_name','surname'])]

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
136,joseph,day,29u - 36u,-,7.7,7.19,NaN


In [217]:
treatments_cut[treatments_cut.duplicated(subset=['given_name','surname'])]

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change


In [218]:
adverse_reactions.duplicated().sum()

np.int64(0)

In [219]:
patients.describe()

,patient_id,weight,height,bmi
count,503.000000,503.000000,503.000000,503.000000
mean,252.000000,173.434990,66.634195,27.483897
std,145.347859,33.916741,4.411297,5.276438
min,1.000000,48.800000,27.000000,17.100000
25%,126.500000,149.300000,63.000000,23.300000
50%,252.000000,175.300000,67.000000,27.200000
75%,377.500000,199.500000,70.000000,31.750000
max,503.000000,255.900000,79.000000,37.700000


In [220]:
patients[patients['weight']==48.800000]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
210,211,female,Camilla,Zaitseva,4689 Briarhill Lane,Wooster,OH,44691.0,United States,330-202-2145CamillaZaitseva@superrito.com,11/26/1938,48.8,63,19.1


In [221]:
patients[patients['height']==27.00000]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,334-515-7487TimNeudorf@cuvox.de,2/18/1928,192.3,27,26.1


In [222]:
treatments.describe()

,hba1c_start,hba1c_end,hba1c_change
count,280.000000,280.000000,171.000000
mean,7.985929,7.589286,0.546023
std,0.568638,0.569672,0.279555
min,7.500000,7.010000,0.200000
25%,7.660000,7.270000,0.340000
50%,7.800000,7.420000,0.380000
75%,7.970000,7.570000,0.920000
max,9.950000,9.580000,0.990000


In [223]:
treatments.sort_values('hba1c_change', na_position='first')

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,NaN
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,NaN
8,saber,ménard,-,54u - 54u,8.08,7.70,NaN
9,asia,woźniak,30u - 36u,-,7.76,7.37,NaN
10,joseph,day,29u - 36u,-,7.70,7.19,NaN
...,...,...,...,...,...,...,...
49,jackson,addison,-,42u - 42u,7.99,7.51,0.98
17,gina,cain,-,36u - 36u,7.88,7.40,0.98
138,giovana,rocha,-,23u - 21u,7.87,7.38,0.99
32,laura,ehrlichmann,-,43u - 40u,7.95,7.46,0.99


In [224]:
treatments_cut.describe()

,hba1c_start,hba1c_end,hba1c_change
count,70.000000,70.000000,42.000000
mean,7.838000,7.443143,0.518810
std,0.423007,0.418706,0.270719
min,7.510000,7.020000,0.280000
25%,7.640000,7.232500,0.340000
50%,7.730000,7.345000,0.370000
75%,7.860000,7.467500,0.907500
max,9.910000,9.460000,0.970000


Data Quality Dimensions
- 
- Completeness -> is data missing?
- Validity -> is data valid (negative height, duplicate patient id)
- Accuracy -> data is valid but not accurate (weight -> 1kg)
- Consistency -> both valid and accurate but written differently (New Youk and NY)

Order of severity
- 
Completeness <- Validity <- Accuracy <- Consistency

Data Cleaning Order
- 
1. Quality -> Completeness
2. Tidiness
3. Quality -> Validity
4. Quality -> Accuracy
5. Quality -> Consistency

Steps involved in Data cleaning
- 
- Define
- Code
- Test

Always make sure to create a copy of your pandas dataframe before you start the cleaning process

In [225]:
patients_df=patients.copy()
treatments_df=treatments.copy()
treatments_cut_df=treatments_cut.copy()
adverse_reactions_df=adverse_reactions.copy()

# Define
- replace all missing values of patients df with no data
- Sub hba1c_start from hba1c_end to get all the change values
- in patients table we will use regex to separate email and phone

In [226]:
# replace all missing values of patients df with no data
patients_df[patients_df['address'].isnull()]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
209,210,female,Lalita,Eldarkhanov,NaN,NaN,NaN,NaN,NaN,NaN,8/14/1950,143.4,62,26.2
219,220,male,Mỹ,Quynh,NaN,NaN,NaN,NaN,NaN,NaN,4/9/1978,237.8,69,35.1
230,231,female,Elisabeth,Knudsen,NaN,NaN,NaN,NaN,NaN,NaN,9/23/1976,165.9,63,29.4
234,235,female,Martina,Tománková,NaN,NaN,NaN,NaN,NaN,NaN,4/7/1936,199.5,65,33.2
242,243,male,John,O'Brian,NaN,NaN,NaN,NaN,NaN,NaN,2/25/1957,205.3,74,26.4
249,250,male,Benjamin,Mehler,NaN,NaN,NaN,NaN,NaN,NaN,10/30/1951,146.5,69,21.6
257,258,male,Jin,Kung,NaN,NaN,NaN,NaN,NaN,NaN,5/17/1995,231.7,69,34.2
264,265,female,Wafiyyah,Asfour,NaN,NaN,NaN,NaN,NaN,NaN,11/3/1989,158.6,63,28.1
269,270,female,Flavia,Fiorentino,NaN,NaN,NaN,NaN,NaN,NaN,10/9/1937,175.2,61,33.1
278,279,female,Generosa,Cabán,NaN,NaN,NaN,NaN,NaN,NaN,12/16/1962,124.3,69,18.4


In [227]:
patients_df.fillna('No Data', inplace=True)

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,California,92390.0,United States,951-719-9170ZoeWellish@superrito.com,7/10/1976,121.7,66,19.6
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,Illinois,61812.0,United States,PamelaSHill@cuvox.de+1 (217) 569-3204,4/3/1967,118.8,66,19.2
2,3,male,Jae,Debord,1493 Poling Farm Road,York,Nebraska,68467.0,United States,402-363-6804JaeMDebord@gustr.com,2/19/1980,177.8,71,24.8
3,4,male,Liêm,Phan,2335 Webster Street,Woodbridge,NJ,7095.0,United States,PhanBaLiem@jourrapide.com+1 (732) 636-8246,7/26/1951,220.9,70,31.7
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,334-515-7487TimNeudorf@cuvox.de,2/18/1928,192.3,27,26.1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,499,male,Mustafa,Lindström,2530 Victoria Court,Milton Mills,ME,3852.0,United States,207-477-0579MustafaLindstrom@jourrapide.com,4/10/1959,181.1,72,24.6
499,500,male,Ruman,Bisliev,494 Clarksburg Park Road,Sedona,AZ,86341.0,United States,928-284-4492RumanBisliev@gustr.com,3/26/1948,239.6,70,34.4
500,501,female,Jinke,de Keizer,649 Nutter Street,Overland Park,MO,64110.0,United States,816-223-6007JinkedeKeizer@teleworm.us,1/13/1971,171.2,67,26.8
501,502,female,Chidalu,Onyekaozulu,3652 Boone Crockett Lane,Seattle,WA,98109.0,United States,ChidaluOnyekaozulu@jourrapide.com1 360 443 2060,2/13/1952,176.9,67,27.7


In [228]:
patients_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    str    
 2   given_name    503 non-null    str    
 3   surname       503 non-null    str    
 4   address       503 non-null    str    
 5   city          503 non-null    str    
 6   state         503 non-null    str    
 7   zip_code      503 non-null    str    
 8   country       503 non-null    str    
 9   contact       503 non-null    str    
 10  birthdate     503 non-null    str    
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(2), int64(2), str(10)
memory usage: 111.4 KB


In [229]:
# Sub hba1c_start from hba1c_end to get all the change values
treatments_df.head()

,given_name,surname,auralin,novodra,hba1c_start,hba1c_end,hba1c_change
0,veronika,jindrová,41u - 48u,-,7.63,7.20,NaN
1,elliot,richardson,-,40u - 45u,7.56,7.09,0.97
2,yukitaka,takenaka,-,39u - 36u,7.68,7.25,NaN
3,skye,gormanston,33u - 36u,-,7.97,7.62,0.35
4,alissa,montez,-,33u - 29u,7.78,7.46,0.32


In [230]:
# code
treatments_df['hba1c_change']=treatments_df['hba1c_start']-treatments_df['hba1c_end']
treatments_cut_df['hba1c_change']=treatments_cut_df['hba1c_start']-treatments_cut_df['hba1c_end']

In [231]:
# test
print(treatments_df.info())
print(treatments_cut_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 280 entries, 0 to 279
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    280 non-null    str    
 1   surname       280 non-null    str    
 2   auralin       280 non-null    str    
 3   novodra       280 non-null    str    
 4   hba1c_start   280 non-null    float64
 5   hba1c_end     280 non-null    float64
 6   hba1c_change  280 non-null    float64
dtypes: float64(3), str(4)
memory usage: 21.8 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 70 entries, 0 to 69
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    70 non-null     str    
 1   surname       70 non-null     str    
 2   auralin       70 non-null     str    
 3   novodra       70 non-null     str    
 4   hba1c_start   70 non-null     float64
 5   hba1c_end     70 non-null     float64
 6   hba1c_change  70 non-null 

In [232]:
# in patients table we will use regex to separate email and phone
import re
def find_contact_details(text:str)->tuple:
    # it the value is NaN, then return it
    if pd.isna(text):
        return np.nan
    
    # Create the phone number pattern
    phone_number_pattern=re.compile(r'(\+[\d]{1,3}\s)?(\(?[\d]{3}\)?\s?-?[\d]{3}\s?-?[\d]{4})')
    # find the phone number from the value/text, as a result we will get a list
    phone_number=re.findall(phone_number_pattern,text)

    # if lenth is 0, then the range can't find any ph number, the define with NaN
    if len(phone_number)<=0:
        phone_number=np.nan
    # if the country code is attached with the ph number, for that case the first
    # element will be the country code and 2nd element will be the actual ph number
    # So, get that ph number
    elif len(phone_number)>=2:
        phone_number=phone_number[1]
    # else, we will get the ph number. Grab it.
    else:
        phone_number=phone_number[0]
    # if we found the ph number(with/wihtout country code), then remove that part from the actual value.
    # after removing the ph number, the remaining string might be the email address.
    possible_email_add=re.sub(phone_number_pattern,"",text).strip()
    # Then return the ph number and the email address
    return phone_number,possible_email_add 

In [233]:
patients_df['phone']=patients_df['contact'].apply(lambda x: find_contact_details(x)).apply(lambda x:x[0])
patients_df['email']=patients_df['contact'].apply(lambda x: find_contact_details(x)).apply(lambda x:x[1])

In [234]:
patients_df.drop(columns='contact', inplace=True)

In [235]:
# Merge treatments and treatments_cut
treatments_df=pd.concat([treatments_df,treatments_cut_df])

In [236]:
# Auralin and novadra col should be split into 2 cols start and end dose
# `-` in novardra and auralin col treated as nan.
treatments_df=treatments_df.melt(id_vars=['given_name','surname','hba1c_start','hba1c_end','hba1c_change'],var_name='type',value_name='dosage_range')

In [237]:
treatments_df=treatments_df[treatments_df['dosage_range'] !='-']

In [238]:
treatments_df['dosage_start']=treatments_df['dosage_range'].str.split('-').str.get(0)
treatments_df['dosage_end']=treatments_df['dosage_range'].str.split('-').str.get(1)

In [239]:
treatments_df.drop(columns='dosage_range',inplace=True)

In [240]:
# remove u from auralin and novadra column.
treatments_df['dosage_start']=treatments_df['dosage_start'].str.replace('u','').astype('int')
treatments_df['dosage_end']=treatments_df['dosage_end'].str.replace('u','').astype('int')

In [241]:
# treatments_df['dosage_start']=treatments_df['dosage_start'].astype('int')
# treatments_df['dosage_end']=treatments_df['dosage_end'].astype('int')

In [242]:
treatments_df.info()

<class 'pandas.DataFrame'>
Index: 350 entries, 0 to 698
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    350 non-null    str    
 1   surname       350 non-null    str    
 2   hba1c_start   350 non-null    float64
 3   hba1c_end     350 non-null    float64
 4   hba1c_change  350 non-null    float64
 5   type          350 non-null    str    
 6   dosage_start  350 non-null    int64  
 7   dosage_end    350 non-null    int64  
dtypes: float64(3), int64(2), str(3)
memory usage: 31.7 KB


In [243]:
# adverse_reactions merge with treatments
treatments_df=treatments_df.merge(adverse_reactions_df,how='left',on=['given_name','surname'])